<a href="https://colab.research.google.com/github/AbdulUMSL/EENG-1108/blob/main/HW5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
// ============================================================
//  ESP32-CAM FAST MJPEG STREAM - IMPROVED v0.3
//  Focus: Network buffer efficiency & Sensor Overclocking
// ============================================================

#include "esp_camera.h"
#include <WiFi.h>
#include <WebServer.h>
#include "esp_wifi.h"
#include "esp_bt.h"

const char* ssid     = "ESP32CAM";
const char* password = "12345678";

WebServer server(80);

// Pin Definitions (AI Thinker)
#define PWDN_GPIO_NUM   32
#define RESET_GPIO_NUM  -1
#define XCLK_GPIO_NUM    0
#define SIOD_GPIO_NUM   26
#define SIOC_GPIO_NUM   27
#define Y9_GPIO_NUM     35
#define Y8_GPIO_NUM     34
#define Y7_GPIO_NUM     39
#define Y6_GPIO_NUM     36
#define Y5_GPIO_NUM     21
#define Y4_GPIO_NUM     19
#define Y3_GPIO_NUM     18
#define Y2_GPIO_NUM      5
#define VSYNC_GPIO_NUM  25
#define HREF_GPIO_NUM   23
#define PCLK_GPIO_NUM   22

#define PART_BOUNDARY "frame"

const char HTML_PAGE[] PROGMEM = R"(
<!DOCTYPE html><html><head><meta name="viewport" content="width=device-width,initial-scale=1">
<style>body{margin:0;background:#000;display:flex;justify-content:center;align-items:center;height:100vh;}
img{width:100vw;max-height:100vh;object-fit:contain;}</style></head>
<body><img src="/stream"></body></html>
)";

void handleRoot() {
    server.send_P(200, "text/html", HTML_PAGE);
}

// ============================================================
//  IMPROVED STREAM HANDLER
// ============================================================
void handleStream() {
    WiFiClient client = server.client();
    client.setNoDelay(true);

    client.print("HTTP/1.1 200 OK\r\n");
    client.print("Content-Type: multipart/x-mixed-replace;boundary=" PART_BOUNDARY "\r\n");
    client.print("Access-Control-Allow-Origin: *\r\n\r\n");

    while (client.connected()) {
        camera_fb_t* fb = esp_camera_fb_get();
        if (!fb) { continue; }

        // Elementary Fix 1: Send the header in ONE print call.
        // Small individual prints trigger multiple tiny TCP packets.
        client.printf("--" PART_BOUNDARY "\r\nContent-Type: image/jpeg\r\nContent-Length: %u\r\n\r\n", fb->len);

        // Elementary Fix 2: Larger buffer chunks.
        // Standardizing on 8KB or using the full buffer if possible reduces loop overhead.
        uint8_t *fbBuf = fb->buf;
        size_t fbLen = fb->len;

        // We write the entire frame at once. The underlying ESP32 stack
        // handles the fragmentation more efficiently than a manual 'while' loop with 4KB.
        client.write(fbBuf, fbLen);

        client.print("\r\n");
        esp_camera_fb_return(fb);

        // Yield to let the WiFi stack breathe without a hard delay
        yield();
    }
}

void setupCamera() {
    camera_config_t cfg;
    cfg.ledc_channel = LEDC_CHANNEL_0;
    cfg.ledc_timer   = LEDC_TIMER_0;
    cfg.pin_d0       = Y2_GPIO_NUM;
    cfg.pin_d1       = Y3_GPIO_NUM;
    cfg.pin_d2       = Y4_GPIO_NUM;
    cfg.pin_d3       = Y5_GPIO_NUM;
    cfg.pin_d4       = Y6_GPIO_NUM;
    cfg.pin_d5       = Y7_GPIO_NUM;
    cfg.pin_d6       = Y8_GPIO_NUM;
    cfg.pin_d7       = Y9_GPIO_NUM;
    cfg.pin_xclk     = XCLK_GPIO_NUM;
    cfg.pin_pclk     = PCLK_GPIO_NUM;
    cfg.pin_vsync    = VSYNC_GPIO_NUM;
    cfg.pin_href     = HREF_GPIO_NUM;
    cfg.pin_sccb_sda = SIOD_GPIO_NUM;
    cfg.pin_sccb_scl = SIOC_GPIO_NUM;
    cfg.pin_pwdn     = PWDN_GPIO_NUM;
    cfg.pin_reset    = RESET_GPIO_NUM;

    // Elementary Fix 3: Increase XCLK to 24MHz (Max stable for most modules)
    cfg.xclk_freq_hz  = 24000000;           // Bumped from 20MHz to 24MHz
    cfg.pixel_format  = PIXFORMAT_JPEG;
    cfg.frame_size    = FRAMESIZE_QVGA;     // 320x240 is the sweet spot

    // Elementary Fix 4: Lower JPEG quality slightly (Higher number = lower quality)
    // Quality 10-12 is sharp, but 20-25 is MUCH faster for video.
    cfg.jpeg_quality  = 20;

    cfg.fb_count      = 2;                  // Use 2 for lower memory latency
    cfg.grab_mode     = CAMERA_GRAB_LATEST;
    cfg.fb_location   = CAMERA_FB_IN_PSRAM;

    if (esp_camera_init(&cfg) != ESP_OK) {
        Serial.println("Camera init failed!");
        return;
    }

    sensor_t* s = esp_camera_sensor_get();
    // Optimization: Disable features that require heavy image processing
    s->set_whitebal(s, 1);
    s->set_awb_gain(s, 1);
    s->set_aec2(s, 0);           // Disable "night mode" AEC to keep frame rate high in low light
}

void setup() {
    Serial.begin(115200);

    // Force CPU to Max speed
    setCpuFrequencyMhz(240);

    // Elementary Fix 5: Ensure Bluetooth is TOTALLY off.
    // This reduces heat and power fluctuations that cause frame drops.
    btStop();

    setupCamera();

    WiFi.mode(WIFI_AP);
    // Disable Power Save for WiFi - Keeps the "radio" awake constantly
    esp_wifi_set_ps(WIFI_PS_NONE);

    WiFi.softAP(ssid, password);

    server.on("/", handleRoot);
    server.on("/stream", handleStream);
    server.begin();
}

void loop() {
    server.handleClient();
}